# Search Algorithms: One Consistent Style

Every algorithm uses the **same structure as your A\* code**:

| Piece | How it is stored |
|---|---|
| Graph | `graph = {node: {neighbour: edge_cost}}` |
| Heuristic | `h = {node: estimate}` |
| Parents | `parent = {node: previous_node}` (start is `None`) |
| Cost so far | `g_cost = {node: cost}` |
| Order of expansion | `explored = []` |

Every function is called as `algorithm(start, goal)` and prints **Explored Nodes, Final path, Total path cost**.

The only real difference between algorithms is **which node comes out of the frontier next**:

| Algorithm | Frontier | Takes next |
|---|---|---|
| BFS | queue (`deque`) | oldest |
| DFS | stack (`list`) | newest |
| UCS | heap | lowest `g` |
| A* | heap | lowest `g + h` |
| DLS | recursion | DFS, but never deeper than `limit` |
| IDS | recursion | DLS with limit 0, 1, 2, ... |
| IDA* | recursion | DFS, but never beyond `f = g + h > threshold` |

**Run the cells in order.**

## Setup: graph, heuristic, and one shared print function

In [1]:
from collections import deque
import heapq

graph = {
    'S' : {'A' : 2, 'B': 5},
    'A' : {'C' : 2, 'D': 4},
    'B' : {'D' : 1},
    'C' : {'G' : 5},
    'D' : {'G' : 2},
    'G' : {}
}

h = {
    'S' : 7,
    'A' : 4,
    'B' : 3,
    'C' : 3,
    'D' : 2,
    'G' : 0
}

In [2]:
def print_solution(goal, explored, parent, g_cost):
    # Same path-rebuilding code as your A* version, shared by every algorithm
    if goal not in explored:
        print("Goal not found")
        return

    path = []
    node = goal

    while node is not None:
        path.append(node)
        node = parent[node]

    path.reverse()

    print(f"Explored Nodes: {explored}")
    print(f"Final path: {path}")
    print(f"Total path cost: {g_cost[goal]}")

## 1. BFS: oldest node first
Uses a queue. It finds the path with the **fewest edges**, which is not always the cheapest. `parent` also works as the "seen" check, so each node is only added once.

In [3]:
def bfs(start, goal):
    queue = deque([start])
    explored = []
    parent = {start : None}
    g_cost = {start : 0}

    while queue:
        current = queue.popleft()          # oldest first

        explored.append(current)
        

        if current == goal:
            break

        for neighbour, edge_cost in graph[current].items():
            if neighbour not in parent:    # never add the same node twice
                parent[neighbour] = current
                g_cost[neighbour] = g_cost[current] + edge_cost
                queue.append(neighbour)

    print_solution(goal, explored, parent, g_cost)

bfs('S', 'G') 

Explored Nodes: ['S', 'A', 'B', 'C', 'D', 'G']
Final path: ['S', 'A', 'C', 'G']
Total path cost: 9


## 2. DFS: newest node first
Uses a stack. The stack holds `(node, who_discovered_it)`. We record the parent when a node is **popped** the first time, so the path follows the real DFS route and a later push cannot overwrite it.

In [4]:
def dfs(start, goal):
    stack = [(start, None)]
    explored = []
    parent = {}
    g_cost = {}

    while stack:
        current, came_from = stack.pop()   # newest first

        if current in explored:
            continue

        explored.append(current)
        parent[current] = came_from
        g_cost[current] = 0 if came_from is None else g_cost[came_from] + graph[came_from][current]

        if current == goal:
            break

        # reversed so the FIRST neighbour is the one popped first
        for neighbour in reversed(list(graph[current])):
            if neighbour not in explored:
                stack.append((neighbour, current))

    print_solution(goal, explored, parent, g_cost)

dfs('S', 'G')

Explored Nodes: ['S', 'A', 'C', 'G']
Final path: ['S', 'A', 'C', 'G']
Total path cost: 9


## 3. UCS: cheapest-so-far first
Identical to your A\* code, except the heap is ordered by `g` alone (no `h`). It always finds the cheapest path.

In [5]:
def ucs(start, goal):
    pq = [(0, start)]
    explored = []
    parent = {start : None}
    g_cost = {start : 0}

    while pq:
        cost, current = heapq.heappop(pq)

        if cost != g_cost[current]:        # stale entry
            continue

        if current in explored:
            continue

        explored.append(current)

        if current == goal:
            break

        for neighbour, edge_cost in graph[current].items():
            new_cost = g_cost[current] + edge_cost

            if neighbour not in g_cost or new_cost < g_cost[neighbour]:
                g_cost[neighbour] = new_cost
                parent[neighbour] = current
                heapq.heappush(pq, (new_cost, neighbour))

    print_solution(goal, explored, parent, g_cost)

ucs('S', 'G')

Explored Nodes: ['S', 'A', 'C', 'B', 'D', 'G']
Final path: ['S', 'A', 'D', 'G']
Total path cost: 8


## 4. A*: lowest g + h first
Your code, unchanged except it now uses the shared `print_solution`.

In [6]:
def a_star(start, goal):
    pq = [(h[start], 0, start)]
    explored = []
    parent = {start : None}
    g_cost = {start : 0}

    while pq:
        f, cost, current = heapq.heappop(pq)

        if cost != g_cost[current]:
            continue

        if current in explored:
            continue

        explored.append(current)

        if current == goal:
            break

        for neighbour, edge_cost in graph[current].items():
            new_cost = g_cost[current] + edge_cost

            if neighbour not in g_cost or new_cost < g_cost[neighbour]:
                g_cost[neighbour] = new_cost
                parent[neighbour] = current

                f_cost = new_cost + h[neighbour]
                heapq.heappush(pq, (f_cost, new_cost, neighbour))

    print_solution(goal, explored, parent, g_cost)

a_star('S', 'G')

Explored Nodes: ['S', 'A', 'C', 'B', 'D', 'G']
Final path: ['S', 'A', 'D', 'G']
Total path cost: 8


In [7]:
def befs(start, goal):
    queue = deque([start])
    explored = []
    parent = {start: None}
    g_cost = {start: 0}

    while queue:
        current = queue.popleft()

        explored.append(current)

        if current == goal:
            break

        for neighbour, edge_cost in graph[current].items():
            if neighbour not in parent:
                parent[neighbour] = current
                g_cost[neighbour] = g_cost[current] + edge_cost
                queue.append(neighbour)

    print_solution(goal, explored, parent, g_cost)

def defs(start, goal):
    stack = [(start, None)]
    explored = []
    parent = {}
    g_cost = {}

    while stack:
        current, came_from = stack.pop()

        if current in explored:
            continue

        explored.append(current)
        parent[current] = came_from
        g_cost[current] = 0 if came_from is None else g_cost[came_from] + graph[came_from][current]

        if current == goal:
            break

        for neighbour in reversed(list(graph[current])):
            if neighbour not in explored:
                stack.append((neighbour, current))

    print_solution(goal, explored, parent, g_cost)

def uecs(start, goal):
    pq = [(0, start)]
    explored = []
    parent = {start: None}
    g_cost = {start: 0}

    while pq:
        cost, current = heapq.heappop(pq)

        if cost != g_cost[current]:
            continue

        if current in explored:
            continue

        explored.append(current)
        if current == goal:
            break

        for neighbour, edge_cost in graph[current].items():
            new_cost = g_cost[current] + edge_cost

            if neighbour not in g_cost or new_cost < g_cost[neighbour]:
                g_cost[neighbour] = new_cost
                parent[neighbour] = current

                heapq.heappush(pq, (new_cost, neighbour))

    print_solution(goal, explored, parent, g_cost)

def astar(start, goal):
    pq = [(h[start], 0, start)]
    explored = []
    parent = {start: None}
    g_cost = {start: 0}

    while pq:
        f, cost, current = heapq.heappop(pq)

        if cost != g_cost[current]:
            continue

        if current in explored:
            continue

        explored.append(current)

        if current == goal:
            break

        for neighbour, edge_cost in graph[current].items():
            new_cost = g_cost[current] + edge_cost

            if neighbour not in g_cost or new_cost < g_cost[neighbour]:
                g_cost[neighbour] = new_cost
                parent[neighbour] = current
                f_cost = new_cost + h[neighbour]
                heapq.heappush(pq, (f_cost, new_cost, neighbour))

    print_solution(goal, explored, parent, g_cost)

def gbfs(start, goal):
    pq = [(h[start], start)]
    explored = []
    parent = {start: None}
    g_cost = {start: 0}

    while pq:
        cost, current = heapq.heappop(pq)
        
        if current in explored:
            continue

        explored.append(current)

        if current == goal:
            break

        for neighbour, edge_cost in graph[current].items():
            new_cost = g_cost[current] + edge_cost

            if neighbour not in g_cost or new_cost < g_cost[neighbour]:
                parent[neighbour] = current
                g_cost[neighbour] = new_cost

                heapq.heappush(pq, (h[neighbour], neighbour))

    print_solution(goal, explored, parent, g_cost)

gbfs('S', 'G')

Explored Nodes: ['S', 'B', 'D', 'G']
Final path: ['S', 'B', 'D', 'G']
Total path cost: 8


## 5. DLS: DFS with a depth limit
Recursive, so no stack variable: Python's call stack does that job. `parent` and `g_cost` are set **just before** going down into a neighbour, so when the goal is found the parent chain is exactly the route taken.

`depth_limited` does the work and returns one of:
- `"found"`
- `"cutoff"`: stopped at the limit, a solution *might* be deeper
- `"failure"`: nothing here at all

`dls` calls it once and prints. (IDS in the next cell reuses `depth_limited`.)

In [8]:
def depth_limited(start, goal, limit):
    explored = []
    parent = {start : None}
    g_cost = {start : 0}
    on_path = {start}                      # avoids going round in circles

    def recurse(current, depth):
        explored.append(current)

        if current == goal:
            return "found"

        if depth == limit:
            return "cutoff"

        cutoff_happened = False

        for neighbour, edge_cost in graph[current].items():
            if neighbour in on_path:
                continue

            parent[neighbour] = current
            g_cost[neighbour] = g_cost[current] + edge_cost

            on_path.add(neighbour)
            result = recurse(neighbour, depth + 1)
            on_path.remove(neighbour)

            if result == "found":
                return "found"
            if result == "cutoff":
                cutoff_happened = True

        return "cutoff" if cutoff_happened else "failure"

    result = recurse(start, 0)
    return result, explored, parent, g_cost


def dls(start, goal, limit):
    result, explored, parent, g_cost = depth_limited(start, goal, limit)

    print(f"Depth limit: {limit}")
    if result == "cutoff":
        print("Cutoff: limit too small, a solution might be deeper")
        return

    print_solution(goal, explored, parent, g_cost)

dls('S', 'G', 2)      # too small
print()
dls('S', 'G', 3)      # just enough

Depth limit: 2
Cutoff: limit too small, a solution might be deeper

Depth limit: 3
Explored Nodes: ['S', 'A', 'C', 'G']
Final path: ['S', 'A', 'C', 'G']
Total path cost: 9


## 6. IDS: run DLS with limit 0, 1, 2, ...
Stops at the first limit that does not get cut off.

In [9]:
def ids(start, goal, max_depth=50):
    for limit in range(max_depth + 1):
        result, explored, parent, g_cost = depth_limited(start, goal, limit)

        if result == "cutoff":
            print(f"Limit {limit}: cutoff")
            continue

        print(f"Limit {limit}: {result}")
        print_solution(goal, explored, parent, g_cost)
        return

    print("Goal not found")

ids('S', 'G')

Limit 0: cutoff
Limit 1: cutoff
Limit 2: cutoff
Limit 3: found
Explored Nodes: ['S', 'A', 'C', 'G']
Final path: ['S', 'A', 'C', 'G']
Total path cost: 9


## 7. IDA*: like IDS, but the cap is on f = g + h
Start with `threshold = h[start]`. Each round is a DFS that refuses to go where `f > threshold`. If the goal is not found, the next threshold is the **smallest f that went over**.

In [10]:
def ida_star(start, goal):
    explored = []
    parent = {}
    g_cost = {}
    on_path = set()

    def search(current, threshold):
        f = g_cost[current] + h[current]

        if f > threshold:
            return f                       # report how far over we went

        explored.append(current)

        if current == goal:
            return "found"

        smallest_over = float("inf")

        for neighbour, edge_cost in graph[current].items():
            if neighbour in on_path:
                continue

            parent[neighbour] = current
            g_cost[neighbour] = g_cost[current] + edge_cost

            on_path.add(neighbour)
            result = search(neighbour, threshold)
            on_path.remove(neighbour)

            if result == "found":
                return "found"
            smallest_over = min(smallest_over, result)

        return smallest_over

    threshold = h[start]

    while True:
        explored.clear()
        parent.clear()
        g_cost.clear()
        parent[start] = None
        g_cost[start] = 0
        on_path.clear()
        on_path.add(start)

        result = search(start, threshold)
        print(f"Threshold {threshold}: explored {explored}")

        if result == "found":
            break

        if result == float("inf"):
            print("Goal not found")
            return

        threshold = result                 # raise the cap and go again

    print_solution(goal, explored, parent, g_cost)

ida_star('S', 'G')

Threshold 7: explored ['S', 'A', 'C']
Threshold 8: explored ['S', 'A', 'C', 'D', 'G']
Explored Nodes: ['S', 'A', 'C', 'D', 'G']
Final path: ['S', 'A', 'D', 'G']
Total path cost: 8


## Compare them all
Look at **Explored Nodes** (how much work) and **Total path cost** (how good the answer). BFS and DFS ignore edge costs, so they can return a more expensive path.

In [11]:
for name, algo in [("BFS", bfs), ("DFS", dfs), ("UCS", ucs), ("A*", a_star), ("IDA*", ida_star)]:
    print(f"----- {name} -----")
    algo('S', 'G')
    print()

----- BFS -----
Explored Nodes: ['S', 'A', 'B', 'C', 'D', 'G']
Final path: ['S', 'A', 'C', 'G']
Total path cost: 9

----- DFS -----
Explored Nodes: ['S', 'A', 'C', 'G']
Final path: ['S', 'A', 'C', 'G']
Total path cost: 9

----- UCS -----
Explored Nodes: ['S', 'A', 'C', 'B', 'D', 'G']
Final path: ['S', 'A', 'D', 'G']
Total path cost: 8

----- A* -----
Explored Nodes: ['S', 'A', 'C', 'B', 'D', 'G']
Final path: ['S', 'A', 'D', 'G']
Total path cost: 8

----- IDA* -----
Threshold 7: explored ['S', 'A', 'C']
Threshold 8: explored ['S', 'A', 'C', 'D', 'G']
Explored Nodes: ['S', 'A', 'C', 'D', 'G']
Final path: ['S', 'A', 'D', 'G']
Total path cost: 8



## Try it yourself
1. **Change a cost** in `graph` (for example `'S': {'A': 2, 'B': 1}`) and re-run. Which algorithms change their answer?
2. **Break the heuristic**: set `h['A'] = 100`. A* is no longer guaranteed to find the cheapest path. Does it still?
3. **Add a loop**: add `'D': {'G': 2, 'S': 1}`. The `on_path` / `explored` checks should keep everything from running forever.
4. **Trace by hand** first, then check against the printed `Explored Nodes`.

In [12]:
from collections import deque
import heapq

In [13]:
def print_sol(goal, explored, parent, cost):
    if goal not in explored:
        print("Goal not found")
        return

    node = goal
    path = []

    while node is not None:
        path.append(node)
        node = parent[node]

    path.reverse()

    print(f"Explored Nodes: {explored}")
    print(f"Final Path: {path}")
    print(f"Total Cost: {cost[goal]}")

In [56]:
farmer = farmer_prob()
queen = queen_prob()
eight = eight_puzzle()
def astar(problem):
    graph = problem["successors"]
    h = problem["h"]
    goal = problem["goal"]
    start = problem["start"]
    is_goal = problem["is_goal"]

    pq = [(h(start), 0, start)]
    parent = {start: None}
    g_cost = {start: 0}
    explored = []
    
    while pq:
        f, cost, current = heapq.heappop(pq)
        if cost != g_cost[current]:
            continue

        if current in explored:
            continue

        explored.append(current)

        if is_goal(current):
            goal = current
            break

        for neighbour, edge_cost in graph(current):
            new_cost = g_cost[current] + edge_cost

            if neighbour not in g_cost or new_cost < g_cost[neighbour]:
                g_cost[neighbour] = new_cost
                parent[neighbour] = current
                
                f_cost = new_cost + h(neighbour)

                heapq.heappush(pq, (f_cost, new_cost, neighbour))

    print_solution(goal, explored, parent, g_cost)

astar(eight)

Explored Nodes: [(2, 1, 3, 4, 7, 6, 5, 8, 0), (2, 1, 3, 4, 7, 0, 5, 8, 6), (2, 1, 3, 4, 7, 6, 5, 0, 8), (2, 1, 3, 4, 0, 6, 5, 7, 8), (2, 1, 3, 4, 7, 6, 0, 5, 8), (2, 1, 0, 4, 7, 3, 5, 8, 6), (2, 1, 3, 4, 0, 7, 5, 8, 6), (2, 0, 3, 4, 1, 6, 5, 7, 8), (2, 1, 3, 0, 4, 6, 5, 7, 8), (2, 1, 3, 0, 7, 6, 4, 5, 8), (2, 1, 3, 4, 6, 0, 5, 7, 8), (0, 2, 3, 4, 1, 6, 5, 7, 8), (2, 1, 3, 5, 4, 6, 0, 7, 8), (2, 1, 3, 7, 0, 6, 4, 5, 8), (2, 1, 3, 5, 4, 6, 7, 0, 8), (2, 1, 3, 7, 5, 6, 4, 0, 8), (2, 1, 3, 5, 4, 6, 7, 8, 0), (2, 1, 3, 7, 5, 6, 4, 8, 0), (2, 0, 1, 4, 7, 3, 5, 8, 6), (2, 0, 3, 4, 1, 7, 5, 8, 6), (2, 1, 3, 0, 4, 7, 5, 8, 6), (2, 1, 3, 4, 8, 7, 5, 0, 6), (0, 1, 3, 2, 4, 6, 5, 7, 8), (0, 1, 3, 2, 7, 6, 4, 5, 8), (0, 2, 1, 4, 7, 3, 5, 8, 6), (0, 2, 3, 4, 1, 7, 5, 8, 6), (2, 1, 0, 4, 6, 3, 5, 7, 8), (2, 1, 3, 4, 6, 8, 5, 7, 0), (2, 1, 3, 4, 8, 7, 0, 5, 6), (2, 1, 3, 5, 4, 7, 0, 8, 6), (2, 3, 0, 4, 1, 6, 5, 7, 8), (1, 0, 3, 2, 4, 6, 5, 7, 8), (1, 0, 3, 2, 7, 6, 4, 5, 8), (2, 0, 3, 7, 1, 6, 4, 5, 8

In [58]:
def farmer_prob():
    names = ['Farmer', 'Wolf', 'Sheep', 'Grass']

    def successor(state):
        result = []
        farmer = state[0]

        for item in [None, 1, 2, 3]:
            if item is not None and state[item] != farmer:
                continue

            new = list(state)
            new[0] = 1 - farmer

            if item is not None:
                new[item] = 1 - farmer

            new = tuple(new)

            if is_safe(new):
                result.append((new, 1))
                
        return result

    def is_safe(state):
        farmer, wolf, sheep, grass = state
        if wolf == sheep and farmer != wolf:
            return False
        
        if grass == sheep and farmer != grass:
            return False

        return True
    
    return {
        "start" : (0,0,0,0),
        "goal" : (1,1,1,1),
        "successors" : successor,
        "h" : lambda s: sum(1 for x in s[1:] if x == 0),
        "is_goal" : lambda s: s==(1,1,1,1)
    }

In [60]:
def queen_prob(n=8):

    def successor(state):
        row = len(state)
        result = []

        for col in range(n):
            if all(col != c and abs(col-c) != row-r for r,c in enumerate(state)):
                result.append((state+ (col,), 1))

        return result

    return{
        "start" : (),
        "goal" : lambda s: len(s)==n,
        "successors" : successor,
        "is_goal": lambda s: len(s) == n,
        "h" : lambda s: n - len(s)
    }

In [61]:
def eight_puzzle(start=(2, 1, 3, 4, 7, 6, 5, 8, 0)):

    goal=(1, 2, 3, 4, 5, 6, 7, 8, 0)
    goal_pos = {tile: divmod(i,3) for i, tile in enumerate(goal)}
    
    def successor(state):
        result = []
        b = state.index(0)
        r, c = divmod(b, 3)

        for dr, dc in [(1,0), (-1,0), (0, 1), (0, -1)]:
            nr, nc = dr+r, dc+c

            if 0 <= nr < 3 and 0 <= nc < 3:
                n = nr*3+nc
                new = list(state)
                new[b], new[n] = new[n], new[b]
                result.append((tuple(new), 1))

        return result

    def h2(state):
        total = 0
        for i, tile in enumerate(state):
            if tile != 0:
                r, c = divmod(i, 3)
                gr, gc = goal_pos[tile]
                total += abs(gr-r) + abs(gc-c)
        return total
    
    return{
        "start" : start,
        "goal" : goal,
        "successors" : successor,
        "is_goal" : lambda s : s==goal,
        "h" : h2
    }